In [2]:
import os
import sys
import logging
import datetime
import numpy as np
import cv2
import mediapipe as mp
from pyts.image import MarkovTransitionField

In [3]:
DATASET_PATH = r"C:\Users\hp\OneDrive\Desktop\CESI_UL Internship\Dataset"
OUTPUT_PATH  = r"C:\Users\hp\StressDetection\DL\outputttt"

# Your assigned participants
PARTICIPANTS = [55, 56]


TASKS = ["T1", "T2", "T3"]


In [4]:
WINDOW_SIZE     = 140
STEP            = 35
MTF_IMAGE_SIZE  = 64
MTF_N_BINS      = 4
MTF_CHANNELS    = 5
BLINK_THRESHOLD = 0.2
MIN_DETECT_RATE = 0.50

# 15 graph node landmark IDs (stress-relevant, excludes normalization anchors)
GRAPH_NODE_IDS = [33, 133, 145, 153, 158, 159,   # EAR region
                  474, 475, 476, 477,              # Iris / gaze
                  70,                              # Brow
                  13, 14, 61, 291]                 # Lips

In [5]:
os.makedirs(OUTPUT_PATH, exist_ok=True)

log_path = os.path.join(OUTPUT_PATH, "extract_geo_log.txt")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler(sys.stdout)
    ]
)
log = logging.getLogger()

In [6]:

log.info("=" * 60)
log.info("extract_geo.py — START")
log.info(f"Date/Time     : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log.info(f"Python        : {sys.version}")
log.info(f"NumPy         : {np.__version__}")
log.info(f"MediaPipe     : {mp.__version__}")
log.info(f"OpenCV        : {cv2.__version__}")
try:
    import pyts
    log.info(f"pyts          : {pyts.__version__}")
except Exception:
    log.info("pyts          : version unknown")
log.info(f"Dataset path  : {DATASET_PATH}")
log.info(f"Output path   : {OUTPUT_PATH}")
log.info(f"Participants  : {PARTICIPANTS}")
log.info("=" * 60)

2026-06-14 23:10:16,379 — INFO — ============================================================
2026-06-14 23:10:16,384 — INFO — extract_geo.py — START
2026-06-14 23:10:16,387 — INFO — Date/Time     : 2026-06-14 23:10:16
2026-06-14 23:10:16,389 — INFO — Python        : 3.9.25 (main, Nov  3 2025, 22:44:01) [MSC v.1929 64 bit (AMD64)]
2026-06-14 23:10:16,390 — INFO — NumPy         : 2.0.2
2026-06-14 23:10:16,390 — INFO — MediaPipe     : 0.10.9
2026-06-14 23:10:16,394 — INFO — OpenCV        : 4.13.0
2026-06-14 23:10:16,395 — INFO — pyts          : 0.13.0
2026-06-14 23:10:16,403 — INFO — Dataset path  : C:\Users\hp\OneDrive\Desktop\CESI_UL Internship\Dataset
2026-06-14 23:10:16,405 — INFO — Output path   : C:\Users\hp\StressDetection\DL\outputttt
2026-06-14 23:10:16,405 — INFO — Participants  : [55, 56]
2026-06-14 23:10:16,411 — INFO — ============================================================


In [7]:

def euclidean(p1, p2):
    """2D Euclidean distance between two (x, y) points."""
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


def compute_features(lm):
    """
    Compute 5 stress features from MediaPipe normalized landmarks.
    lm: function lm(i) returns (x, y) in normalized [0,1] coordinates.
    Returns [EAR, Blink, Gaze, Brow_dist, Lip_ratio]
    """

    # EAR — Eye Aspect Ratio
    vertical   = euclidean(lm(153), lm(158)) + euclidean(lm(145), lm(159))
    horizontal = euclidean(lm(33),  lm(133))
    EAR        = vertical / (2.0 * horizontal) if horizontal > 0 else 0.0

    # Blink
    Blink = 1.0 if EAR < BLINK_THRESHOLD else 0.0

    # Gaze — scale-invariant iris position relative to eye center
    eye_center_x = (lm(33)[0] + lm(133)[0]) / 2.0
    iris_x       = np.mean([lm(474)[0], lm(475)[0], lm(476)[0], lm(477)[0]])
    eye_width    = euclidean(lm(33), lm(133))
    Gaze         = (iris_x - eye_center_x) / eye_width if eye_width > 0 else 0.0

    # Brow distance — Euclidean, normalized by face height (matches ML formula)
    face_h   = euclidean(lm(10),  lm(152))
    brow_eye = euclidean(lm(70),  lm(159))
    Brow     = brow_eye / face_h if face_h > 0 else 0.0

    # Lip ratio
    lip_v = euclidean(lm(13), lm(14))
    lip_h = euclidean(lm(61), lm(291))
    Lip   = lip_v / lip_h if lip_h > 0 else 0.0

    return [EAR, Blink, Gaze, Brow, Lip]


In [10]:
def extract_graph_nodes(landmarks):
    """
    Extract 15 graph node coordinates as normalized (x, y, z).
    Returns array of shape (15, 3).
    """
    coords = []
    for idx in GRAPH_NODE_IDS:
        lm = landmarks[idx]
        coords.append([lm.x, lm.y, lm.z])
    return np.array(coords, dtype=np.float32)


def slice_into_windows(buffer, window_size, step):
    """
    Slice a buffer (list of frames) into overlapping windows.
    Leftover frames that don't fill a complete window are discarded.
    """
    windows = []
    total   = len(buffer)
    for start in range(0, total - window_size + 1, step):
        windows.append(buffer[start : start + window_size])
    return windows


def convert_to_mtf(feature_windows):
    """
    Convert feature windows to MTF images.
    Input  : list of windows, each shape (140, 5)
    Output : numpy array of shape (num_windows, 64, 64, 5)
    """
    mtf_transform = MarkovTransitionField(
        image_size=MTF_IMAGE_SIZE,
        n_bins=MTF_N_BINS,
        strategy='uniform'
    )
    mtf_windows = []

    for window in feature_windows:
        window_arr = np.array(window, dtype=np.float32)  # (140, 5)
        channel_images = []

        for ch in range(MTF_CHANNELS):
            series = window_arr[:, ch].reshape(1, -1)    # (1, 140)

            # clip to avoid constant series issues with MTF
            series = np.clip(series, -1e6, 1e6)
            if np.std(series) < 1e-8:
                series = series + np.random.normal(0, 1e-8, series.shape)

            mtf_img = mtf_transform.fit_transform(series)  # (1, 64, 64)
            channel_images.append(mtf_img[0])              # (64, 64)

        stacked = np.stack(channel_images, axis=-1)        # (64, 64, 5)
        mtf_windows.append(stacked)

    return np.array(mtf_windows, dtype=np.float32)         # (N, 64, 64, 5)


def sanity_check(path, expected_shape_suffix, label):
    """
    Reload a saved .npy file and verify shape and NaN absence.
    expected_shape_suffix: tuple of dims after dim 0, e.g. (140, 15, 3)
    """
    try:
        arr = np.load(path)
        assert arr.shape[0] > 0, "zero windows"
        for i, dim in enumerate(expected_shape_suffix):
            assert arr.shape[i + 1] == dim, \
                f"shape[{i+1}] expected {dim}, got {arr.shape[i+1]}"
        assert not np.isnan(arr).any(), "NaN values found"
        log.info(f"  Sanity OK  [{label}] shape={arr.shape}")
        return True
    except Exception as e:
        log.warning(f"  Sanity FAIL [{label}] {e}")
        return False

In [11]:

mp_face_mesh = mp.solutions.face_mesh

for participant_id in PARTICIPANTS:

    p_str = str(participant_id)
    p_out = os.path.join(OUTPUT_PATH, f"p{p_str}")
    os.makedirs(p_out, exist_ok=True)

    # ── Crash recovery ──────────────────────────────────────────────────────
    all_files = [
        os.path.join(p_out, f"p{p_str}_T{t}_{ftype}.npy")
        for t in ["1", "2", "3"]
        for ftype in ["graph", "mtf"]
    ]
    if all(os.path.exists(f) for f in all_files):
        log.info(f"Participant {p_str} — all 6 files exist, skipping")
        continue

    log.info(f"\nParticipant {p_str} — starting extraction")

    # ── Task loop ────────────────────────────────────────────────────────────
    for task in TASKS:

        task_num = task[1]  # "1", "2", "3"

        graph_path = os.path.join(p_out, f"p{p_str}_T{task_num}_graph.npy")
        mtf_path   = os.path.join(p_out, f"p{p_str}_T{task_num}_mtf.npy")

        # Individual task crash recovery
        if os.path.exists(graph_path) and os.path.exists(mtf_path):
            log.info(f"  {task} — files exist, skipping")
            continue

        # ── Open video ───────────────────────────────────────────────────────
        video_path = os.path.join(
            DATASET_PATH,
            f"Participant{participant_id}",
            f"vid_s{participant_id}_{task}.avi"
        )

        if not os.path.exists(video_path):
            log.warning(f"  {task} — video not found: {video_path}")
            continue

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            log.warning(f"  {task} — could not open video: {video_path}")
            continue

        # ── Initialize buffers ───────────────────────────────────────────────
        landmarks_buffer = []
        features_buffer  = []
        last_landmarks   = None
        last_features    = None
        failed_frames    = 0
        total_frames     = 0

        # ── PART 1 — Frame loop ──────────────────────────────────────────────
        with mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as face_mesh:

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                total_frames += 1
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results   = face_mesh.process(rgb_frame)

                if results.multi_face_landmarks:
                    face_lm = results.multi_face_landmarks[0].landmark

                    # lm(i) returns normalized (x, y) tuple
                    def lm(i):
                        return (face_lm[i].x, face_lm[i].y)

                    # Extract 15 graph nodes (x, y, z normalized)
                    graph_row = extract_graph_nodes(face_lm)

                    # Compute 5 features
                    try:
                        feat_row = compute_features(lm)
                    except Exception as e:
                        log.warning(f"  Frame {total_frames} feature error: {e}")
                        feat_row = last_features if last_features is not None \
                                   else [0.0] * 5

                    landmarks_buffer.append(graph_row)
                    features_buffer.append(feat_row)

                    last_landmarks = graph_row
                    last_features  = feat_row

                else:
                    # Face not detected — forward fill
                    failed_frames += 1
                    if last_landmarks is not None:
                        landmarks_buffer.append(last_landmarks)
                        features_buffer.append(last_features)
                    else:
                        landmarks_buffer.append(np.zeros((15, 3), dtype=np.float32))
                        features_buffer.append([0.0] * 5)

        cap.release()

        # ── Quality check ─────────────────────────────────────────────────────
        detection_rate = (total_frames - failed_frames) / total_frames \
                         if total_frames > 0 else 0.0

        log.info(f"  {task} — frames={total_frames}, "
                 f"failed={failed_frames}, "
                 f"detection_rate={detection_rate:.3f}")

        if detection_rate < MIN_DETECT_RATE:
            log.warning(f"  {task} — detection rate {detection_rate:.3f} "
                        f"below {MIN_DETECT_RATE} — FLAGGED, skipping")
            continue

        # ── PART 2 — Windowing ────────────────────────────────────────────────
        landmark_windows = slice_into_windows(landmarks_buffer, WINDOW_SIZE, STEP)
        feature_windows  = slice_into_windows(features_buffer,  WINDOW_SIZE, STEP)

        num_windows = len(landmark_windows)
        log.info(f"  {task} — windows produced: {num_windows}")

        if num_windows == 0:
            log.warning(f"  {task} — zero windows produced, skipping")
            continue

        # ── PART 3 — MTF conversion ───────────────────────────────────────────
        log.info(f"  {task} — converting to MTF images...")
        mtf_array = convert_to_mtf(feature_windows)
        # mtf_array shape: (num_windows, 64, 64, 5)

        # ── PART 4 — Save ─────────────────────────────────────────────────────
        graph_array = np.array(landmark_windows, dtype=np.float32)
        # graph_array shape: (num_windows, 140, 15, 3)

        np.save(graph_path, graph_array)
        np.save(mtf_path,   mtf_array)

        log.info(f"  {task} — saved graph {graph_array.shape} → {graph_path}")
        log.info(f"  {task} — saved mtf   {mtf_array.shape}   → {mtf_path}")

        # ── PART 5 — Sanity check ─────────────────────────────────────────────
        sanity_check(graph_path, (140, 15, 3), "graph")
        sanity_check(mtf_path,   (64, 64, 5),  "mtf")

log.info("\n" + "=" * 60)
log.info("extract_geo.py — COMPLETE")
log.info("=" * 60)


2026-06-14 23:11:44,547 — INFO — 
Participant 55 — starting extraction
2026-06-14 23:17:51,865 — INFO —   T1 — frames=6325, failed=0, detection_rate=1.000
2026-06-14 23:17:51,865 — INFO —   T1 — windows produced: 177
2026-06-14 23:17:51,865 — INFO —   T1 — converting to MTF images...
2026-06-14 23:17:58,274 — INFO —   T1 — saved graph (177, 140, 15, 3) → C:\Users\hp\StressDetection\DL\outputttt\p55\p55_T1_graph.npy
2026-06-14 23:17:58,274 — INFO —   T1 — saved mtf   (177, 64, 64, 5)   → C:\Users\hp\StressDetection\DL\outputttt\p55\p55_T1_mtf.npy
2026-06-14 23:17:58,318 — INFO —   Sanity OK  [graph] shape=(177, 140, 15, 3)
2026-06-14 23:17:58,350 — INFO —   Sanity OK  [mtf] shape=(177, 64, 64, 5)
2026-06-14 23:24:15,521 — INFO —   T2 — frames=6325, failed=0, detection_rate=1.000
2026-06-14 23:24:15,533 — INFO —   T2 — windows produced: 177
2026-06-14 23:24:15,536 — INFO —   T2 — converting to MTF images...
2026-06-14 23:24:16,321 — INFO —   T2 — saved graph (177, 140, 15, 3) → C:\Users\